In [4]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch import optim, nn, utils, Tensor
from torch.utils.data import DataLoader, TensorDataset

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint

In [5]:
and_input = torch.Tensor([[0.,0.], [0.,1.], [1.,0.], [1., 1.]])
and_labels = torch.Tensor([[0.],[0.], [0.], [1.]])

In [6]:
# TensorDataset: Dataset wrapping tensors.
# Each sample will be retrieved by indexing tensors along the first dimension.
and_data = TensorDataset(and_input, and_labels)

# Data loader combines a dataset and a sampler, and provides an iterable over the given dataset.
train_loader = DataLoader(and_data, batch_size = 4, shuffle=True)

for data in train_loader.dataset:
    print(data)

(tensor([0., 0.]), tensor([0.]))
(tensor([0., 1.]), tensor([0.]))
(tensor([1., 0.]), tensor([0.]))
(tensor([1., 1.]), tensor([1.]))


In [7]:
class XORModel(L.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.history = {
            'epochs': [],
            'loss': []
        }
    
    def forward(self, x):
        """Passes a tensor through the layers of a NN.

        The forward method acts as a mapper or medium where data is passed
        between multiple layers and the activation function.

        Args:
            x: Input tensor.
        """
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        """ Performs the training of the NN.

        Args:
            batch: Data that is being passed in the data loader is accessed
            in batches. This consists of two items: one is the input/features
            data, and the other item is targets.
            batch_idx: This is the index number or the sequence number
            for the batch of data.
        """
        x, y = batch
        y_hat = self.forward(x)

        loss = nn.functional.mse_loss(y_hat, y)

        self.history['epochs'].append(self.current_epoch)
        self.history['loss'].append(loss.item())

        if self.current_epoch % 100 == 0:
            print(f'Epoch: {self.current_epoch}, Loss: {loss}')
        
        self.log('train_loss', loss)
        return loss
    
    def configure_optimizers(self):
        """ Defines the optimizer for the model
        """
        optimizer = optim.Adam(self.parameters(), lr = 1e-2)
        return optimizer


In [8]:
xor_layers = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1),
    nn.Sigmoid()
)

xor_model = XORModel(xor_layers)

checkpoint_callback = ModelCheckpoint()

trainer = L.Trainer(max_epochs=1000, callbacks=[checkpoint_callback])

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [9]:
trainer.fit(xor_model, train_loader)

2024-10-08 17:53:38.015424: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-08 17:53:38.117210: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-08 17:53:38.150541: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-08 17:53:38.160720: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-08 17:53:38.225078: I tensorflow/core/platform/cpu_feature_guar

Epoch 999: 100%|██████████| 1/1 [00:00<00:00, 176.79it/s, v_num=0]

`Trainer.fit` stopped: `max_epochs=1000` reached.


Epoch 999: 100%|██████████| 1/1 [00:00<00:00, 127.25it/s, v_num=0]
